In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Running on device: {device}")


Running on device: cpu


In [2]:

# ResNet expects images to be slightly larger, 
# so we resize them from 32x32 up to 224x224)
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Standard ImageNet colors
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
# Note: Using a smaller batch size (32) because 224x224 images take up more memory!
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True) 

# ==========================================
# 2. TRANSFER LEARNING
# ==========================================
print("Downloading the brain of ResNet-18...")
# Download the architecture AND the pre-trained weights
ai_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# FREEZING THE BRAIN:
for param in ai_model.parameters():
    param.requires_grad = False

# SWAPIING THE HEAD: 
# Finding out how many neurons were feeding into ResNet's final layer
num_features = ai_model.fc.in_features 
# Replacing the final 1000-category layer with a new 10-category layer for CIFAR
ai_model.fc = nn.Linear(num_features, 10) 

# Move the customized model to our CPU/GPU
ai_model = ai_model.to(device)


100.0%
c:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\dell/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


In [3]:
# 3. THE RULES (Only pass the parameters of the NEW layer to the optimizer!)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(ai_model.fc.parameters(), lr=0.001)

# 4. TRAINING LOOP
epochs = 1  
print("\n--- Starting Transfer Learning ---")
for epoch in range(epochs):
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = ai_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Print update every 50 batches so you can see it working
        if i % 50 == 49:
            print(f"Batch {i+1} | Loss: {running_loss/50:.4f}")
            running_loss = 0.0

print("Training Complete!")


--- Starting Transfer Learning ---
Batch 50 | Loss: 1.8919
Batch 100 | Loss: 1.2888
Batch 150 | Loss: 1.0957
Batch 200 | Loss: 0.9945
Batch 250 | Loss: 0.8903
Batch 300 | Loss: 0.8824
Batch 350 | Loss: 0.7966
Batch 400 | Loss: 0.7759
Batch 450 | Loss: 0.7809
Batch 500 | Loss: 0.7933
Batch 550 | Loss: 0.7483
Batch 600 | Loss: 0.7670
Batch 650 | Loss: 0.7439
Batch 700 | Loss: 0.7330
Batch 750 | Loss: 0.6894
Batch 800 | Loss: 0.7253
Batch 850 | Loss: 0.7128
Batch 900 | Loss: 0.6920
Batch 950 | Loss: 0.7449


KeyboardInterrupt: 